In [ ]:
from collections import Counter, defaultdict
import numbers
from pathlib import Path

import fitz
import numpy as np
import pandas as pd

dev_set_base_dir = "path/to/dev-set-100"
dev_set_wald_wvc_base_dir = "path/to/dev-set-Wald-WVC"

In [ ]:
df = pd.read_json("../data/interim/faktencheck-db/faktenscheck_core_corrected.jsonl", lines=True)
print(df.head())

In [ ]:
counters = defaultdict(Counter)
totals = defaultdict(int)
numeric_sums = defaultdict(float)


def process(col_name, value):
    """Flattens lists/dicts and updates counters."""
    if value is None:
        return

    # dict → rekursiv flatten
    if isinstance(value, dict):
        for k, v in value.items():
            process(f"{col_name}.{k}", v)
        return

    # list → jedes Element einzeln
    if isinstance(value, list):
        for item in value:
            process(col_name, item)
        return

    # primitive value
    counters[col_name][value] += 1
    totals[col_name] += 1

    if isinstance(value, numbers.Number):
        numeric_sums[col_name] += value


for col in df.columns:
    for value in df[col]:
        process(col, value)


# Output
print("\n=== Unique gold entries (support) in the dev-set-100 reference data ===")
print(f"Unique docs with annotations: {df.notna().any(axis=1).sum()}")
for col in sorted(counters):
    print(f"\n=== {col} ===")
    print(f"Total values: {totals[col]}")

    if numeric_sums[col] != 0:
        print(f"Numeric sum: {numeric_sums[col]}")

    # for val, count in counters[col].most_common():
    #    print(f"  {val}: {count}")

In [ ]:
df = pd.read_csv(
    "../data/external/organism_trends/Weighted Vote Count Wald Literatur - Sheet1.csv"
)
print(df.head())
df = df[df["Key"] != "#NV"]

In [ ]:
counters = defaultdict(Counter)
totals = defaultdict(int)
numeric_sums = defaultdict(float)
for col in df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]].columns:
    for value in df[col]:
        process(col, value)


# Output
print("\n=== Unique gold entries (support) in the dev-set-Wald-WVC reference data ===")
print(f"Unique docs with annotations: {len(df["Key"].unique())}")
for col in sorted(counters):
    print(f"\n=== {col} ===")
    print(f"Total values: {totals[col]}")

    if numeric_sums[col] != 0:
        print(f"Numeric sum: {numeric_sums[col]}")

    for val, count in counters[col].most_common():
        print(f"  {val}: {count}")

In [ ]:
df = pd.read_json("../data/processed/faktencheck/dev-set-100/predictions.jsonl", lines=True)
print(df.head())

In [ ]:
def count_words(df, col):
    word_counts = []

    for value in df[col].dropna():
        if not isinstance(value, str):
            continue

        words = value.split()  # whitespace split
        word_counts.append(len(words))

    # Summary stats
    sum_words = np.sum(word_counts) if word_counts else 0
    avg_words = np.mean(word_counts) if word_counts else 0
    med_words = np.median(word_counts) if word_counts else 0
    max_words = max(word_counts) if word_counts else 0
    min_words = min(word_counts) if word_counts else 0

    print(f"Column: {col}")
    print(f"Average words per doc: {avg_words:.2f}")
    print(f"Median words per doc: {med_words}")
    print(f"Max words per doc: {max_words}")
    print(f"Min words per doc: {min_words}")
    print(f"Total words: {sum_words}")


col = "text"
count_words(df, col)

In [ ]:
def count_pages(df, col, base_dir):
    page_counts = []

    for file_path in df[col].dropna().unique():
        try:
            doc = fitz.open(Path(base_dir, file_path))
            num_pages = doc.page_count
            page_counts.append(num_pages)
            doc.close()

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    sum_pages = np.sum(page_counts) if page_counts else 0
    avg_pages = np.mean(page_counts) if page_counts else 0
    med_pages = np.median(page_counts) if page_counts else 0
    max_pages = max(page_counts) if page_counts else 0
    min_pages = min(page_counts) if page_counts else 0

    print(f"Average pages per PDF: {avg_pages:.2f}")
    print(f"Median pages per PDF: {med_pages}")
    print(f"Max pages: {max_pages}")
    print(f"Min pages: {min_pages}")
    print(f"Total pages: {sum_pages}")


col = "file_name"
count_pages(df, col, dev_set_base_dir)

In [ ]:
df = pd.read_json(
    "../data/processed/faktencheck/dev-set-Wald-WVC/predictions.jsonl.gz", lines=True
)
print(df.head())

In [ ]:
col = "text"
count_words(df, col)

In [ ]:
col = "file_name"
count_pages(df, col, dev_set_wald_wvc_base_dir)